# Embedding Similarity — Stratified Permutation Tests

**Goal**: Compare the effect of **few-shot** vs **one-shot** vs **zero-shot** prototype construction for embedding similarity experiments.  
Each **model** acts as a **stratum**: within each model-stratum the three domains (medical, administrative, education) share the same embedding space / prototypes, so stratifying by model controls for model-level confounds.

**Three comparisons**
- `one_shot  vs zero_shot`  
- `few_shot  vs one_shot`  
- `few_shot  vs zero_shot`

**Strata**: one stratum per embedding model.  
Within each stratum the correctness table rows come from the union of all three domain test sets; each experiment (one domain × one shot-level) becomes a column.

**Correctness** is binary exact-match (1 if prediction vector == ground-truth vector).  
Predictions are regenerated from the embeddings CSV + prototypes stored in `Parameters`.

> **Locally** only `e5-small` embeddings exist; all cells gracefully skip missing model files.

## 0. Imports & setup

In [ ]:
import os, sys, ast, warnings
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# ── project root ──────────────────────────────────────────────────────────────
project_root = os.path.abspath(".")
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

from orgpackage.aux import load_experiments
from orgpackage.config import DOMAIN_CLASSES_CORR
from orgpackage.tester import stratified_permutation_test

warnings.filterwarnings("ignore")
print("Setup done. Project root:", project_root)

## 1. Load & inspect similarity experiments

In [ ]:
EXPERIMENTS_PATH = "./results/experiments.csv"
EMBEDDINGS_DIR   = "./results/embeddings"

exps = load_experiments(EXPERIMENTS_PATH)
sim_exps = exps[
    (exps["Technique"] == "embedding") &
    (exps["Method"]    == "similarity")
].copy()

print(f"Total similarity experiments: {len(sim_exps)}")

display_df = sim_exps[["ID", "Domain", "Accuracy", "F1"]].copy()
display_df["model"]  = sim_exps["Parameters"].apply(
    lambda p: p.get("model",  "?") if isinstance(p, dict) else "?")
display_df["n_shot"] = sim_exps["Parameters"].apply(
    lambda p: p.get("n_shot", "?") if isinstance(p, dict) else "?")
display_df

## 2. Group experiments by model × n_shot

In [ ]:
# model_shot_index  →  {model: {n_shot: [exp_id, ...]}}
model_shot_index = {}
for _, row in sim_exps.iterrows():
    params = row["Parameters"]
    if not isinstance(params, dict):
        continue
    model  = params.get("model")
    n_shot = params.get("n_shot")
    model_shot_index.setdefault(model, {}).setdefault(n_shot, []).append(row["ID"])

print("Models found:", list(model_shot_index.keys()))
for model, shots in model_shot_index.items():
    print(f"\n  {model}")
    for shot, ids in shots.items():
        print(f"    {shot:12s}: {ids}")

## 3. Helper — build binary correctness table for one model

For a given model:
1. Load `results/embeddings/{model}_embeddings.csv`.
2. For every experiment using this model, re-apply the similarity rule  
   (`threshold = 1 − distance`, prototypes from `Parameters["prototypes"]`).
3. Return a DataFrame: rows = instance URIs, columns = experiment IDs, values ∈ {0, 1}.

In [ ]:
def _classify_instance(x_emb, prototypes, threshold, structure, classes):
    """Reproduce evaluate_similarity_experiment for one instance."""
    is_multiclass = (structure == "2-multiclass")
    x = np.array(x_emb).reshape(1, -1)
    similarities = {}

    for cls in classes:
        proto = prototypes.get(cls)
        best_sim = -1.0
        if proto is None:
            similarities[cls] = -1.0
            continue

        if isinstance(proto, dict):      # few-shot: country-keyed dict
            for _, p in proto.items():
                if p is None:
                    continue
                if isinstance(p, list):
                    p = np.array(p)
                if not isinstance(p, np.ndarray) or np.isnan(p).any():
                    continue
                try:
                    sim = cosine_similarity(x, p.reshape(1, -1))[0][0]
                    best_sim = max(best_sim, sim)
                except Exception:
                    pass
        else:                            # 0-shot / 1-shot: single prototype
            if isinstance(proto, list):
                proto = np.array(proto)
            if isinstance(proto, np.ndarray) and not np.isnan(proto).any():
                try:
                    best_sim = cosine_similarity(x, proto.reshape(1, -1))[0][0]
                except Exception:
                    pass
        similarities[cls] = best_sim

    preds = {cls: 0 for cls in classes}
    if all(v == -1.0 for v in similarities.values()):
        return preds

    if is_multiclass:
        for cls in classes:
            if similarities.get(cls, -1.0) >= threshold:
                preds[cls] = 1
    else:
        valid = {k: v for k, v in similarities.items() if v != -1.0}
        if valid:
            best_cls = max(valid, key=valid.get)
            if valid[best_cls] >= threshold:
                preds[best_cls] = 1
    return preds


def build_embedding_correctness_table(model_name, exp_ids, sim_exps_df,
                                       embeddings_dir=EMBEDDINGS_DIR):
    """
    Build binary correctness table for all exp_ids that use `model_name`.

    Rows    = instances (indexed by 'instance' URI)
    Columns = experiment IDs
    Values  = 1 if predicted labels == ground-truth labels, else 0
    """
    emb_path = os.path.join(embeddings_dir, f"{model_name}_embeddings.csv")
    if not os.path.exists(emb_path):
        raise FileNotFoundError(f"Embeddings not found: {emb_path}")

    print(f"  Loading embeddings: {emb_path}")
    emb_df = pd.read_csv(emb_path, low_memory=False)
    emb_col = f"{model_name}_embedding"

    def _parse_emb(v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return None
        if isinstance(v, str):
            try:
                return np.array(ast.literal_eval(v))
            except Exception:
                return None
        return v if isinstance(v, np.ndarray) else None

    emb_df[emb_col] = emb_df[emb_col].apply(_parse_emb)
    valid_mask = emb_df[emb_col].apply(
        lambda v: v is not None and not np.isnan(v).any()
    )
    emb_df = emb_df[valid_mask].reset_index(drop=True)
    print(f"    {len(emb_df)} instances with valid embeddings.")

    correctness_cols = {}

    for exp_id in exp_ids:
        exp_row = sim_exps_df[sim_exps_df["ID"] == exp_id]
        if exp_row.empty:
            print(f"    WARNING: {exp_id} not found — skipping.")
            continue
        exp        = exp_row.iloc[0]
        domain     = exp["Domain"]
        params     = exp["Parameters"]
        classes    = DOMAIN_CLASSES_CORR[domain]
        threshold  = 1.0 - params["distance"]
        prototypes = params["prototypes"]
        structure  = params["structure"]

        missing = [c for c in classes if c not in emb_df.columns]
        if missing:
            print(f"    WARNING: class columns {missing} missing for {exp_id}.")
            continue

        domain_df = emb_df[emb_df[classes].notna().all(axis=1)].copy()

        hit = []
        for _, row in domain_df.iterrows():
            y_true = {cls: int(row[cls]) for cls in classes}
            preds  = _classify_instance(
                row[emb_col], prototypes, threshold, structure, classes
            )
            hit.append((row["instance"],
                        int(all(preds[cls] == y_true[cls] for cls in classes))))

        s = pd.Series([h[1] for h in hit],
                      index=[h[0] for h in hit],
                      name=exp_id, dtype=int)
        correctness_cols[exp_id] = s
        print(f"    {exp_id} ({domain:15s}): {len(s)} rows, "
              f"mean correctness = {s.mean():.3f}")

    if not correctness_cols:
        raise ValueError(f"No predictions generated for model '{model_name}'.")

    return pd.DataFrame(correctness_cols).fillna(0).astype(int)


print("Helpers defined.")

## 4. Build correctness tables — one per model (= one stratum)

In [ ]:
strata_by_model = {}   # {model_name: correctness_df}

for model_name, shot_groups in model_shot_index.items():
    emb_path = os.path.join(EMBEDDINGS_DIR, f"{model_name}_embeddings.csv")
    if not os.path.exists(emb_path):
        print(f"[SKIP] {model_name}: {emb_path} not found.")
        continue

    all_exp_ids = [eid for ids in shot_groups.values() for eid in ids]
    print(f"\n{'='*60}")
    print(f"Model: {model_name}  |  experiments: {all_exp_ids}")

    try:
        ct = build_embedding_correctness_table(model_name, all_exp_ids, sim_exps)
        strata_by_model[model_name] = ct
        print(f"  → Correctness table shape: {ct.shape}")
    except Exception as e:
        print(f"  ERROR: {e}")

print("\nModels with correctness tables:", list(strata_by_model.keys()))

## 5. Define experiment pairs for the three comparisons

In [ ]:
SHOT_PAIRS = [
    ("1_shot",   "0_shot",   "one_shot vs zero_shot"),
    ("few_shot", "1_shot",   "few_shot vs one_shot"),
    ("few_shot", "0_shot",   "few_shot vs zero_shot"),
]

DOMAINS = ["medical", "administrative", "education"]

def get_exp_id(msi, model, n_shot, domain):
    """Return the experiment ID matching (model, n_shot, domain)."""
    ids = msi.get(model, {}).get(n_shot, [])
    prefix = domain[:3]
    for eid in ids:
        if eid.startswith(prefix):
            return eid
    return None

# Validation printout
for shot_a, shot_b, label in SHOT_PAIRS:
    print(f"\n  [{label}]")
    for model in strata_by_model:
        for domain in DOMAINS:
            id_a = get_exp_id(model_shot_index, model, shot_a, domain)
            id_b = get_exp_id(model_shot_index, model, shot_b, domain)
            flag = "\u2713" if (id_a and id_b) else "\u2717"
            print(f"    {flag} {model:20s} / {domain:15s}: {id_a} vs {id_b}")

## 6. Run stratified permutation tests

**Stratification**: one stratum = one embedding model.  
Within each stratum we extract the two relevant columns (A = shot_a, B = shot_b) per domain and pool them.  
We run:
- **per-domain** tests (one stratum per model, rows from that domain only)
- **all-domain pooled** test (all three domains concatenated per model-stratum)

In [ ]:
def run_comparison(shot_a, shot_b, label, strata_by_model, msi,
                   n_perm=10_000):
    results = []

    # ── per-domain ────────────────────────────────────────────────────────────
    for domain in DOMAINS:
        strata = []
        for model, ct in strata_by_model.items():
            id_a = get_exp_id(msi, model, shot_a, domain)
            id_b = get_exp_id(msi, model, shot_b, domain)
            if not id_a or not id_b:
                continue
            if id_a not in ct.columns or id_b not in ct.columns:
                print(f"  WARNING: {id_a} or {id_b} missing in {model} table.")
                continue
            sub = ct[[id_a, id_b]].rename(columns={id_a: "A", id_b: "B"})
            strata.append(sub)

        if not strata:
            continue

        obs, p_val = stratified_permutation_test(
            strata=strata, exp_a="A", exp_b="B",
            n_perm=n_perm, statistic="pooled"
        )
        all_a = np.concatenate([s["A"].values for s in strata])
        all_b = np.concatenate([s["B"].values for s in strata])
        results.append({
            "comparison":  label,
            "scope":       domain,
            "shot_a":      shot_a, "shot_b": shot_b,
            "n_strata":    len(strata),
            "n_instances": sum(len(s) for s in strata),
            "mean_a":      round(all_a.mean(), 4),
            "mean_b":      round(all_b.mean(), 4),
            "obs_diff":    round(obs, 4),
            "p_value":     p_val,
        })

    # ── all-domain pooled ─────────────────────────────────────────────────────
    pooled_strata = []
    for model, ct in strata_by_model.items():
        domain_subs = []
        for domain in DOMAINS:
            id_a = get_exp_id(msi, model, shot_a, domain)
            id_b = get_exp_id(msi, model, shot_b, domain)
            if id_a and id_b and id_a in ct.columns and id_b in ct.columns:
                sub = ct[[id_a, id_b]].rename(columns={id_a: "A", id_b: "B"})
                domain_subs.append(sub)
        if domain_subs:
            pooled_strata.append(pd.concat(domain_subs))

    if pooled_strata:
        obs_all, p_all = stratified_permutation_test(
            strata=pooled_strata, exp_a="A", exp_b="B",
            n_perm=n_perm, statistic="pooled"
        )
        all_a = np.concatenate([s["A"].values for s in pooled_strata])
        all_b = np.concatenate([s["B"].values for s in pooled_strata])
        results.append({
            "comparison":  label,
            "scope":       "all domains",
            "shot_a":      shot_a, "shot_b": shot_b,
            "n_strata":    len(pooled_strata),
            "n_instances": sum(len(s) for s in pooled_strata),
            "mean_a":      round(all_a.mean(), 4),
            "mean_b":      round(all_b.mean(), 4),
            "obs_diff":    round(obs_all, 4),
            "p_value":     p_all,
        })

    return pd.DataFrame(results)


print("run_comparison() defined.")

In [ ]:
N_PERM = 10_000

all_results = []
for shot_a, shot_b, label in SHOT_PAIRS:
    print(f"\n{'='*60}\nRunning: {label}\n{'='*60}")
    df = run_comparison(shot_a, shot_b, label,
                        strata_by_model, model_shot_index, n_perm=N_PERM)
    all_results.append(df)
    print(df.to_string(index=False))

results_df = pd.concat(all_results, ignore_index=True)
print("\nAll tests complete.")

## 7. Apply Holm multiple-testing correction and display final table

In [ ]:
from statsmodels.stats.multitest import multipletests

reject, p_corrected, _, _ = multipletests(results_df["p_value"], method="holm")
results_df["p_corrected"] = p_corrected
results_df["significant"] = reject

pd.set_option("display.max_colwidth", 40)
pd.set_option("display.float_format", "{:.4f}".format)

print("\n=== FULL RESULTS TABLE (Holm-corrected) ===")
display(results_df[[
    "comparison", "scope", "n_strata", "n_instances",
    "mean_a", "mean_b", "obs_diff",
    "p_value", "p_corrected", "significant"
]])

## 8. Save results

In [ ]:
os.makedirs("results", exist_ok=True)
out_path = "results/embedding_permutation_tests.csv"
results_df.to_csv(out_path, index=False)
print(f"Saved to {out_path}")

## 9. Visualisation — bar chart per comparison × domain

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

scope_order = ["medical", "administrative", "education", "all domains"]
comparison_labels = [label for _, _, label in SHOT_PAIRS]

fig, axes = plt.subplots(
    1, len(comparison_labels),
    figsize=(5 * len(comparison_labels), 5),
    sharey=False
)
if len(comparison_labels) == 1:
    axes = [axes]

for ax, comp in zip(axes, comparison_labels):
    sub = results_df[results_df["comparison"] == comp].copy()
    sub = sub.set_index("scope").reindex(
        [s for s in scope_order if s in sub.index]
    )
    colors = ["#d62728" if sig else "#aec7e8" for sig in sub["significant"]]
    bars = ax.bar(range(len(sub)), sub["obs_diff"],
                  color=colors, edgecolor="k", linewidth=0.5)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels(
        [s.replace(" ", "\n") for s in sub.index], fontsize=9
    )
    ax.set_title(comp.replace(" vs ", "\nvs "), fontsize=10, fontweight="bold")
    ax.set_ylabel("Mean correctness diff (A \u2212 B)")

    for bar, (_, row) in zip(bars, sub.iterrows()):
        p_str = (f"p={row['p_corrected']:.3f}"
                 if row['p_corrected'] >= 0.001 else "p<0.001")
        offset = 0.003 if row["obs_diff"] >= 0 else -0.015
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + offset,
                p_str, ha="center", va="bottom", fontsize=7.5)

sig_patch   = mpatches.Patch(color="#d62728", label="Significant (Holm p<0.05)")
insig_patch = mpatches.Patch(color="#aec7e8", label="Not significant")
fig.legend(handles=[sig_patch, insig_patch],
           loc="lower center", ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, -0.05))
fig.suptitle(
    "Embedding Similarity: Stratified Permutation Tests\n"
    "Strata = embedding models  |  Holm correction",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
os.makedirs("figures", exist_ok=True)
fig.savefig("figures/embedding_permutation_tests.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to figures/embedding_permutation_tests.png")

## 10. Per-model diagnostic breakdown

Unadjusted p-values per (model × domain) — diagnostic only, not for reporting.

In [ ]:
per_model_rows = []

for shot_a, shot_b, label in SHOT_PAIRS:
    for model, ct in strata_by_model.items():
        for domain in DOMAINS:
            id_a = get_exp_id(model_shot_index, model, shot_a, domain)
            id_b = get_exp_id(model_shot_index, model, shot_b, domain)
            if not id_a or not id_b:
                continue
            if id_a not in ct.columns or id_b not in ct.columns:
                continue

            a    = ct[id_a].values.astype(float)
            b    = ct[id_b].values.astype(float)
            diff = a - b
            obs  = diff.mean()

            rng   = np.random.default_rng(42)
            signs = rng.choice([-1, 1], size=(N_PERM, len(diff)))
            p_val = max(
                (np.abs((signs * diff).mean(axis=1)) >= abs(obs)).mean(),
                1.0 / N_PERM
            )
            per_model_rows.append({
                "comparison": label, "model": model, "domain": domain,
                "n": len(a),
                "mean_a": round(a.mean(), 4),
                "mean_b": round(b.mean(), 4),
                "obs_diff": round(obs, 4),
                "p_value(unadj)": p_val,
                "id_a": id_a, "id_b": id_b,
            })

per_model_df = pd.DataFrame(per_model_rows)
print("Per-model / per-domain breakdown (unadjusted — diagnostic only):")
display(per_model_df)